* 模块一：十五五战略指数体系_数据清洗与权重分配
* 功能：双文件融合 → MECE去重 → 多维因子评分 → 指数权重计算（含风控约束） → 标准化输出
* 依赖：pandas >= 2.0, numpy >= 1.24, openpyxl

In [16]:
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Tuple, Optional
import warnings

In [23]:
class IndexConstructionPipeline:
    def __init__(self, file1_path: str, file2_path: str):
        self.file1 = Path(file1_path)
        self.file2 = Path(file2_path)
        self.df_raw: Optional[pd.DataFrame] = None
        self.df_scored: Optional[pd.DataFrame] = None
        self.df_weighted: Optional[pd.DataFrame] = None

    def load_and_standardize(self) -> pd.DataFrame:
        """加载双文件并对齐字段，处理规模分类与评分字段映射"""
        df1 = pd.read_excel(self.file1, sheet_name='Sheet1')
        df2 = pd.read_excel(self.file2, sheet_name='Sheet1')

        # 字段标准化
        df1 = df1.rename(columns={'规模分类': '市值规模'})
        df1['政策契合度'] = np.nan
        df1['投资确定性'] = np.nan
        df2['投资风格'] = np.nan

        # 选取核心字段并拼接
        core_cols = ['一级方向', '二级分类', '三级赛道', '标的名称', '代码',
                     '市值规模', '入选说明', '投资风格', '政策契合度', '投资确定性']
        df = pd.concat([df1[core_cols], df2[core_cols]], ignore_index=True)
        df = df.drop_duplicates(subset=['代码', '一级方向', '二级分类', '三级赛道'], keep='last')
        df['代码'] = df['代码'].astype(str).str.strip()
        return df

    def _impute_factor_scores(self, df: pd.DataFrame) -> pd.DataFrame:
        """因子评分插补与标准化（预留API接入接口）"""
        # 政策契合度 & 投资确定性：优先使用文件二显性评分，缺失则按规模/描述代理
        df['政策契合度'] = df['政策契合度'].fillna(
            df.apply(lambda r: 5 if '国家/战略/专精特新/首台套' in str(r['入选说明']) else 4, axis=1)
        )
        df['投资确定性'] = df['投资确定性'].fillna(
            df.apply(lambda r: 5 if '龙头/全球/市占率/量产' in str(r['入选说明']) else 3.5, axis=1)
        )

        # 技术壁垒：代理指标（可替换为研发占比/专利数API）
        df['技术壁垒'] = np.where(df['市值规模'] == '大', 5,
                          np.where(df['市值规模'] == '中', 4, 3))
        
        # 流动性/风格：映射至1-5分
        style_map = {'大': 5.0, '中': 3.5, '小': 2.5}
        df['流动性评分'] = df['市值规模'].map(style_map).fillna(3.0)
        return df

    def apply_scoring_model(self, df: pd.DataFrame) -> pd.DataFrame:
        """计算五维升维模型综合得分"""
        df = self._impute_factor_scores(df)
        w_policy, w_certainty, w_tech, w_liquidity = 0.30, 0.30, 0.20, 0.20
        df['综合得分'] = (
            df['政策契合度'] * w_policy +
            df['投资确定性'] * w_certainty +
            df['技术壁垒'] * w_tech +
            df['流动性评分'] * w_liquidity
        )
        return df.round(2)

    def deduplicate_and_map_tracks(self, df: pd.DataFrame) -> pd.DataFrame:
        """执行主赛道唯一归属去重逻辑"""
        df = df.sort_values(['代码', '综合得分'], ascending=[True, False]).reset_index(drop=True)
        
        def mark_primary_track(group):
            group['主赛道归属标志'] = ['是'] + ['否'] * (len(group) - 1)
            primary_track = group.iloc[0]['三级赛道']
            group['副赛道标签(协同映射)'] = group.apply(
                lambda r: f"原属:{r['三级赛道']}" if r['主赛道归属标志']=='否' else '', axis=1
            )
            # 统一主赛道映射
            group.loc[group['主赛道归属标志']=='否', '三级赛道'] = primary_track
            return group

        df = df.groupby('代码', group_keys=False).apply(mark_primary_track)
        return df

    def calculate_index_weights(self, df: pd.DataFrame, 
                                max_stock_wt: float = 0.05, 
                                max_track_wt: float = 0.15) -> pd.DataFrame:
        """指数权重计算：自由流通市值加权 + 因子调整 + 双重上限约束"""
        # 基础市值权重代理
        mkt_cap_map = {'大': 1.0, '中': 0.6, '小': 0.3}
        df['基础权重'] = df['市值规模'].map(mkt_cap_map).fillna(0.3) * df['综合得分']
        
        # 归一化至1
        df['初始权重'] = df['基础权重'] / df['基础权重'].sum()
        
        # 个股权重上限截断
        df['权重'] = np.minimum(df['初始权重'], max_stock_wt)
        excess = df['权重'].sum() - 1.0
        if excess > 0:
            df.loc[df['权重'] < max_stock_wt, '权重'] -= excess / (df['权重'] < max_stock_wt).sum()
            
        # 单赛道权重上限校验（迭代收敛）
        for _ in range(3):
            track_wt = df.groupby('三级赛道')['权重'].transform('sum')
            over_track = track_wt > max_track_wt
            if over_track.any():
                scale_factor = max_track_wt / track_wt[over_track].iloc[0]
                df.loc[over_track, '权重'] *= scale_factor
            df['权重'] = df['权重'] / df['权重'].sum()
            
        df['指数池标识'] = np.where(df['综合得分'] >= 4.5, '成分股', 
                           np.where(df['综合得分'] >= 4.0, '备选池', '观察池'))
        return df

    def run(self, output_path: str = '十五五十大投资方向_升维体系融合版.xlsx') -> pd.DataFrame:
        """全链路执行"""
        self.df_raw = self.load_and_standardize()
        self.df_scored = self.apply_scoring_model(self.df_raw)
        self.df_weighted = self.calculate_index_weights(self.df_scored)
        self.df_weighted = self.deduplicate_and_map_tracks(self.df_weighted)
        
        # 字段重排与输出
        export_cols = ['一级方向', '二级分类', '三级赛道', '标的名称', '代码', '市值规模',
                       '主赛道归属标志', '副赛道标签(协同映射)', '政策契合度', '投资确定性',
                       '技术壁垒', '流动性评分', '综合得分', '权重', '指数池标识', '入选说明']
        out_df = self.df_weighted[export_cols]
        out_df.to_excel(output_path, index=False)
        print(f"[INFO] 数据清洗与权重分配完成，已输出至: {output_path}")
        return out_df

In [ ]:
pipeline = IndexConstructionPipeline('十大投资方向分类体系94赛道_标的明细331_1.xlsx',
                                     '十大投资方向分类体系111赛道_标的明细190.xlsx')
df_final = pipeline.run()

In [8]:
import plotly.express as px

df = px.data.tips()

# Treemap - 层级关系
fig = px.treemap(
    df, 
    path=['day', 'time', 'sex'], 
    values='total_bill',
    title='Treemap - 层级关系'
)
fig.show()

# Sunburst - 层级关系（环形）
fig = px.sunburst(
    df, 
    path=['day', 'time', 'sex'], 
    values='total_bill',
    title='Sunburst - 层级关系'
)
fig.show()